# B4：多租戶 SaaS Agent 架構設計

> **場景：** B2B SaaS 公司，提供 AI 客服 Agent 給 1,000 家企業客戶使用，每家客戶有自己的知識庫和資料。  
> **核心挑戰：** 如何在共享基礎設施上確保客戶間的資料完全隔離，同時控制成本？

## 核心架構決策

| 決策 | 選擇 | 拒絕的方案 |
|------|------|------------|
| 知識庫隔離 | Shared Vector DB + namespace | 每個租戶獨立 Vector DB |
| Token 限流 | Token-Aware Rate Limiting | Request-based Rate Limiting |
| 成本分攤 | 按 Token 計費到租戶 | 平均分攤 |
| 租戶識別 | JWT + 中間件注入 | 讓 LLM 自己判斷 |
| 知識庫更新 | 異步 Indexing Pipeline | 同步更新 |

In [ ]:
import time
import asyncio
import hashlib
import json
from typing import Optional
from dataclasses import dataclass, field
from collections import defaultdict

print("場景：B2B SaaS，1000 家租戶")
print("每日總請求：500K（平均 500/租戶）")
print("每請求平均：2,500 tokens")

## 決策 1：共享 vs 獨立 Vector DB

**❌ 被拒絕：每個租戶獨立 Vector DB**
- 1,000 個 Vector DB 實例 × $50/月 = $50,000/月
- 維護 1,000 個實例的 ops 成本極高
- 每個實例的資料量很小，資源浪費

**✅ 選擇：Shared Vector DB + Namespace/Tenant 隔離**

In [ ]:
@dataclass
class VectorDocument:
    doc_id: str
    tenant_id: str          # 強制隔離的關鍵欄位
    content: str
    embedding: list[float]
    metadata: dict = field(default_factory=dict)


class MultiTenantVectorStore:
    """
    共享向量資料庫 + 租戶命名空間隔離
    
    為什麼不給每個租戶獨立 DB？
    成本比較：
      獨立 DB：1000 租戶 × $50/月 = $50,000/月
      共享 DB：1 個 DB，按使用量計費 ~$2,000/月
      節省：96%
    
    什麼情況下應該用獨立 DB？
    → 客戶有嚴格的資料主權要求（資料不能與他人共用基礎設施）
    → 大型企業客戶，資料量超過 1M 文件（共享 DB 效能下降）
    → 合規要求：客戶需要獨立審計（VPC-SC 隔離）
    
    隔離機制：
    → Pinecone：namespace 參數（每次讀寫都帶 namespace=tenant_id）
    → Vertex AI Vector Search：index partition by tenant_id
    → pgvector：Row-Level Security on tenant_id column
    """
    
    def __init__(self):
        self._store: dict[str, list[VectorDocument]] = defaultdict(list)
    
    def upsert(self, doc: VectorDocument):
        """寫入時必須帶 tenant_id"""
        # 在生產中：Pinecone 的 namespace=doc.tenant_id
        self._store[doc.tenant_id].append(doc)
    
    def search(self, tenant_id: str, query_embedding: list[float],
               top_k: int = 3, metadata_filter: dict = None) -> list[VectorDocument]:
        """
        搜尋時強制過濾 tenant_id
        即使 query 本身沒有帶 tenant_id，中間件也會注入
        """
        # 只看這個租戶的文件（核心隔離保證）
        tenant_docs = self._store.get(tenant_id, [])
        
        # Metadata 過濾
        if metadata_filter:
            tenant_docs = [
                d for d in tenant_docs
                if all(d.metadata.get(k) == v for k, v in metadata_filter.items())
            ]
        
        # 模擬相似度排序（生產中用真實向量計算）
        return tenant_docs[:top_k]


# 模擬三個租戶的資料
vector_store = MultiTenantVectorStore()

tenant_docs = [
    ("tenant_A", "退換貨政策：7天無理由退換",      {"doc_type": "policy"}),
    ("tenant_A", "退貨流程：需填寫申請表",           {"doc_type": "process"}),
    ("tenant_B", "退款規定：14天內申請",             {"doc_type": "policy"}),
    ("tenant_B", "客服聯繫：週一至週五 9-18 時",    {"doc_type": "contact"}),
    ("tenant_C", "維修保固：一年保固",               {"doc_type": "warranty"}),
]

for tenant_id, content, meta in tenant_docs:
    vector_store.upsert(VectorDocument(
        doc_id=hashlib.md5(content.encode()).hexdigest()[:8],
        tenant_id=tenant_id,
        content=content,
        embedding=[0.1] * 768,
        metadata=meta
    ))

# 展示隔離效果
print("=" * 55)
print("租戶隔離測試")
print("=" * 55)

for tenant in ["tenant_A", "tenant_B", "tenant_C"]:
    results = vector_store.search(tenant, query_embedding=[0.1]*768)
    print(f"\n[{tenant}] 搜尋結果（{len(results)} 筆）：")
    for r in results:
        print(f"  → {r.content}")

print("\n✅ tenant_A 看不到 tenant_B 和 tenant_C 的資料")
print("   這是靠程式碼層的 tenant_id 過濾保證，不靠 LLM 判斷")

## 決策 2：Token-Aware Rate Limiting — 為什麼不用 RPM？

In [ ]:
class TokenAwareRateLimiter:
    """
    Token-Aware Rate Limiter（滑動視窗 Token 桶）
    
    為什麼不用 RPM（每分鐘請求數）限流？
    AI API 的資源消耗不是按請求數，而是按 Token 數：
      Request A：「你好！」→ 50 tokens
      Request B：「分析這份 200 頁合約」→ 150,000 tokens
    如果只限制 RPM，一個大請求可以吃掉所有其他租戶的配額。
    
    設計：每個租戶有 TPM（每分鐘 Token 數）配額
    實作：Redis INCR + EXPIRE（原子操作，防 Race Condition）
    """
    
    # 租戶方案 → TPM 配額
    PLAN_LIMITS = {
        "starter":      100_000,   # 100K TPM
        "professional": 500_000,   # 500K TPM
        "enterprise":   2_000_000, # 2M TPM
    }
    
    def __init__(self):
        # 模擬 Redis（生產中用 redis-py）
        self._usage: dict[str, dict] = {}  # tenant_id → {tokens, window_start}
    
    def _get_window_key(self) -> int:
        """滑動視窗：目前分鐘"""
        return int(time.time() // 60)
    
    def check_and_consume(self, tenant_id: str, plan: str,
                           estimated_tokens: int) -> dict:
        """
        檢查是否超過 TPM 限制，如果沒有就消耗 Token
        在生產中：這是一個 Redis INCRBY + GET + EXPIRE 的原子操作
        """
        limit = self.PLAN_LIMITS.get(plan, 100_000)
        window = self._get_window_key()
        
        key = f"{tenant_id}_{window}"
        if key not in self._usage:
            self._usage[key] = {"tokens": 0, "window": window}
        
        current_usage = self._usage[key]["tokens"]
        
        if current_usage + estimated_tokens > limit:
            return {
                "allowed": False,
                "reason": "TPM 超限",
                "current_usage": current_usage,
                "limit": limit,
                "retry_after_seconds": 60 - int(time.time() % 60)
            }
        
        self._usage[key]["tokens"] += estimated_tokens
        return {
            "allowed": True,
            "tokens_consumed": estimated_tokens,
            "remaining": limit - self._usage[key]["tokens"]
        }


limiter = TokenAwareRateLimiter()

print("Token-Aware Rate Limiting 測試：")
print("租戶 A（starter plan, 100K TPM）")
print("=" * 55)

requests = [
    ("你好！",                           50),
    ("分析這份合約",                     3000),
    ("查訂單狀態",                        100),
    ("分析 200 頁的財報並做摘要",       80000),  # 大請求
    ("再來一個大請求",                   30000),  # 可能超限
]

for query, tokens in requests:
    result = limiter.check_and_consume("tenant_A", "starter", tokens)
    if result["allowed"]:
        print(f"  ✅ '{query}' ({tokens:,} tokens) | 剩餘: {result['remaining']:,}")
    else:
        print(f"  🚫 '{query}' ({tokens:,} tokens) | {result['reason']} | retry in {result['retry_after_seconds']}s")

print("""

生產實作（Redis）：
  pipeline = redis.pipeline()
  pipeline.incrby(f"{tenant_id}:{window_minute}", estimated_tokens)
  pipeline.expire(f"{tenant_id}:{window_minute}", 60)
  current_usage, _ = pipeline.execute()
  
  if current_usage > limit:
      return 429 Too Many Requests

為什麼用 Redis INCRBY 而不是 SELECT + UPDATE？
→ INCRBY 是原子操作，不會有 Race Condition
→ 100 個並發請求同時扣額度，SELECT+UPDATE 會有重複計算
""")

## 決策 3：成本分攤 + 用量追蹤

In [ ]:
class TenantCostTracker:
    """
    每個租戶的精確成本追蹤
    
    為什麼要追蹤到租戶級別？
    1. 計費：按用量計費的定價模型（超量收費）
    2. 成本分析：找出高消耗租戶，評估定價是否合理
    3. Anomaly Detection：某租戶突然暴增 → 可能是 bug 或被攻擊
    4. 容量規劃：預測未來基礎設施成本
    """
    
    MODEL_COSTS = {
        "gemini-flash":  {"input": 0.075, "output": 0.30},   # per 1M tokens
        "gemini-pro":    {"input": 1.25,  "output": 5.00},
        "embedding":     {"input": 0.025, "output": 0.0},
    }
    
    def __init__(self):
        self._usage: dict[str, dict] = defaultdict(lambda: {
            "requests": 0, "input_tokens": 0, "output_tokens": 0,
            "cost_usd": 0.0, "models_used": defaultdict(int)
        })
    
    def record(self, tenant_id: str, model: str,
               input_tokens: int, output_tokens: int):
        costs = self.MODEL_COSTS.get(model, {"input": 0, "output": 0})
        cost = (input_tokens * costs["input"] + output_tokens * costs["output"]) / 1_000_000
        
        usage = self._usage[tenant_id]
        usage["requests"] += 1
        usage["input_tokens"] += input_tokens
        usage["output_tokens"] += output_tokens
        usage["cost_usd"] += cost
        usage["models_used"][model] += 1
    
    def monthly_report(self) -> list[dict]:
        report = []
        for tenant_id, usage in self._usage.items():
            report.append({
                "tenant_id": tenant_id,
                "requests": usage["requests"],
                "total_tokens": usage["input_tokens"] + usage["output_tokens"],
                "cost_usd": round(usage["cost_usd"], 4),
                "avg_tokens_per_req": (usage["input_tokens"] + usage["output_tokens"]) // max(usage["requests"], 1)
            })
        return sorted(report, key=lambda x: x["cost_usd"], reverse=True)


tracker = TenantCostTracker()

# 模擬一天的用量
import random
random.seed(42)

tenants = {
    "tenant_enterprise_A": {"requests": 5000, "avg_input": 3000, "avg_output": 500, "model": "gemini-pro"},
    "tenant_pro_B":        {"requests": 2000, "avg_input": 2000, "avg_output": 300, "model": "gemini-flash"},
    "tenant_starter_C":    {"requests": 200,  "avg_input": 1000, "avg_output": 200, "model": "gemini-flash"},
    "tenant_pro_D":        {"requests": 3000, "avg_input": 8000, "avg_output": 1000, "model": "gemini-pro"},  # 大請求異常
}

for tid, config in tenants.items():
    for _ in range(config["requests"]):
        tracker.record(
            tid, config["model"],
            config["avg_input"] + random.randint(-500, 500),
            config["avg_output"] + random.randint(-100, 100)
        )

print("月度成本報告（按成本降序）：")
print("=" * 65)
print(f"{'租戶':<25} {'請求數':>8} {'總 Tokens':>12} {'成本(USD)':>12} {'均 Tokens/req':>14}")
print("-" * 65)
for row in tracker.monthly_report():
    flag = " ⚠️" if row["avg_tokens_per_req"] > 5000 else ""
    print(f"  {row['tenant_id']:<23} {row['requests']:>8,} {row['total_tokens']:>12,} {row['cost_usd']:>12.2f} {row['avg_tokens_per_req']:>14,}{flag}")

print("\n⚠️  tenant_pro_D 的平均 Token/req 異常高（可能濫用或 bug）")

In [ ]:
print("""
FDE 面試：多租戶 SaaS Agent 架構決策
═══════════════════════════════════════════════════════

Q: 什麼情況下用共享 Vector DB，什麼情況下用獨立？
A: 共享（Namespace 隔離）：
     - SMB 客戶，資料量小（< 100K 文件/租戶）
     - 對成本敏感，1000 個獨立 DB = $50K/月
     - 沒有嚴格的資料主權要求
   獨立 Vector DB：
     - 大型企業，資料量超過 1M 文件
     - 合規要求：資料不能與他人共用基礎設施
     - 客戶需要獨立 VPC（金融、政府）
   混合策略：SMB 用共享，Enterprise 用獨立（加價）

Q: 為什麼 Token-Aware 限流比 RPM 更合適 AI 系統？
A: AI API 的資源消耗和請求數不成比例。
   「你好」是 50 tokens，「分析 200 頁合約」是 150,000 tokens。
   如果只限制 RPM，一個大請求吃掉所有其他租戶的 API 配額，
   導致 Noisy Neighbor 問題。
   Token-Aware 直接限制資源消耗，才能實現 Fair-Share。
   實作上用 Redis INCRBY（原子操作）避免 Race Condition。

Q: 租戶識別為什麼要在中間件層做，不讓 LLM 自己判斷？
A: 安全設計的黃金法則：租戶邊界不能靠 LLM 決策。
   JWT 解碼後，tenant_id 從 token 中提取，
   中間件層強制注入到所有後續操作（向量搜尋、工具呼叫）。
   LLM 的職責是回答問題，不是判斷「這個請求是誰的」。
   Zero Trust：永遠不信任輸入，永遠在程式碼層驗證。
═══════════════════════════════════════════════════════
""")